### Modules / Packages

In [22]:
import os
from pathlib import Path
import json
import pandas as pd


# Data Quality & Cleaning

This notebook performs data validation and cleaning of the credit applications dataset.
The goal is to ensure data consistency, handle missing values, and prepare the dataset for further bias analysis.

## 1. Initial Data Inspection

We inspect the dataset structure, number of records, and variable types to identify potential data quality issues.

In [23]:
from pathlib import Path
import json
import pandas as pd

with open("../data/raw_credit_applications.json") as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


## 2. Missing values analysis

**Goal:** Quantify missingness and document how we decide to handle missing data.  
**What we measure:** Missing count and missing percentage for each column.  
**Next step:** Based on the results, we may drop columns with extremely high missingness to keep the cleaned dataset usable.

In [24]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)

missing_summary = (
    pd.DataFrame({"missing_count": missing_count, "missing_%": missing_pct})
      .sort_values("missing_%", ascending=False)
)

missing_summary

,missing_count,missing_%
notes,500,99.60
financials.annual_salary,497,99.00
loan_purpose,452,90.04
processing_timestamp,440,87.65
decision.rejection_reason,292,58.17
decision.approved_amount,210,41.83
decision.interest_rate,210,41.83
financials.annual_income,5,1.00
applicant_info.ip_address,5,1.00
applicant_info.ssn,5,1.00


In [25]:
df_clean = df.copy()

HIGH_MISSING_THRESHOLD = 95.0
cols_high_missing = missing_summary[missing_summary["missing_%"] >= HIGH_MISSING_THRESHOLD].index.tolist()

print("Columns to drop (>= 95% missing):", cols_high_missing)

df_clean = df_clean.drop(columns=cols_high_missing)

print("Shape before:", df.shape)
print("Shape after :", df_clean.shape)

Columns to drop (>= 95% missing): ['notes', 'financials.annual_salary']
Shape before: (502, 21)
Shape after : (502, 19)


## 3. Inconsistent data types across records

**Goal:** Ensure numeric and datetime fields have consistent and correct data types.  
**What we measure:** We check current dtypes and quantify how many values fail conversion.  
**Decision:** Convert numeric fields using `pd.to_numeric(errors="coerce")` and datetime fields using `pd.to_datetime(errors="coerce")`.

In [26]:
df_clean.dtypes

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info.full_name             object
applicant_info.email                 object
applicant_info.ssn                   object
applicant_info.ip_address            object
applicant_info.gender                object
applicant_info.date_of_birth         object
applicant_info.zip_code              object
financials.annual_income             object
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.loan_approved                 bool
decision.rejection_reason            object
loan_purpose                         object
decision.interest_rate              float64
decision.approved_amount            float64
dtype: object

In [27]:
numeric_cols = [
    "financials.annual_income",
    "financials.debt_to_income",
    "financials.savings_balance",
    "financials.credit_history_months",
    "decision.approved_amount",
    "decision.interest_rate",
]

for c in numeric_cols:
    if c in df_clean.columns:
        before_na = df_clean[c].isna().sum()
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")
        after_na = df_clean[c].isna().sum()
        print(f"{c}: NA before={before_na}, NA after conversion={after_na}")

# Optional datetime conversion (if you keep processing_timestamp)
if "processing_timestamp" in df_clean.columns:
    before_na = df_clean["processing_timestamp"].isna().sum()
    df_clean["processing_timestamp"] = pd.to_datetime(df_clean["processing_timestamp"], errors="coerce")
    after_na = df_clean["processing_timestamp"].isna().sum()
    print(f"processing_timestamp: NA before={before_na}, NA after conversion={after_na}")

financials.annual_income: NA before=5, NA after conversion=5
financials.debt_to_income: NA before=0, NA after conversion=0
financials.savings_balance: NA before=0, NA after conversion=0
financials.credit_history_months: NA before=0, NA after conversion=0
decision.approved_amount: NA before=210, NA after conversion=210
decision.interest_rate: NA before=210, NA after conversion=210
processing_timestamp: NA before=440, NA after conversion=440


## 4. Categorical consistency: gender

**Goal:** Standardize gender values because the dataset contains multiple encodings (e.g., "Female"/"Male" and "F"/"M").  
**What we measure:** Value counts before and after standardization.  
**Decision:** Map known variants to canonical labels ("Female", "Male"). Unknown or inconsistent values are set to missing.

In [28]:
df_clean["applicant_info.gender"].value_counts(dropna=False)

applicant_info.gender
Male      195
Female    193
F          58
M          53
            2
NaN         1
Name: count, dtype: int64

In [29]:
g = df_clean["applicant_info.gender"].astype("string").str.strip().str.lower()

gender_map = {
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female",
}

df_clean["applicant_info.gender"] = g.map(gender_map)

In [30]:
df_clean["applicant_info.gender"].value_counts(dropna=False)

applicant_info.gender
Female    251
Male      248
NaN         3
Name: count, dtype: int64

## 5. Date formats and derived feature: age

**Goal:** Parse `applicant_info.date_of_birth` consistently and compute an `age` column.  
**What we measure:**  
- Number of invalid/unparseable birth dates  
- Number of out-of-range ages (e.g., negative or unrealistically high)  
**Decision:**  
- Invalid dates → missing age  
- Ages outside a realistic range (0–120) → set to missing

In [31]:
dob = pd.to_datetime(df_clean["applicant_info.date_of_birth"], errors="coerce")

invalid_dob = dob.isna().sum()
print("Invalid/unparseable date_of_birth:", invalid_dob)

# Fixed reference date for reproducibility
ref_date = pd.Timestamp("2024-01-15")

df_clean["age"] = ((ref_date - dob).dt.days // 365).astype("Int64")

# Remove impossible ages
out_of_range = ((df_clean["age"] < 0) | (df_clean["age"] > 120)).sum()
print("Out-of-range ages:", out_of_range)

df_clean.loc[(df_clean["age"] < 0) | (df_clean["age"] > 120), "age"] = pd.NA

df_clean["age"].describe()

Invalid/unparseable date_of_birth: 162
Out-of-range ages: 0


count        340.0
mean     38.935294
std      11.149901
min           21.0
25%          30.75
50%           37.0
75%           45.0
max           65.0
Name: age, dtype: Float64

**Note:** A subset of `date_of_birth` values cannot be parsed reliably with the available information.
For those records we set `age` to missing to avoid introducing incorrect ages through assumptions about date formats.

In [32]:
df_clean["age"].isna().sum(), df_clean["age"].head(10)

(np.int64(162),
 0      22
 1      31
 2      34
 3      40
 4      24
 5    <NA>
 6    <NA>
 7      32
 8      33
 9      34
 Name: age, dtype: Int64)

## 6. Invalid or impossible numeric values

**Goal:** Detect values that are logically impossible (e.g., negative credit history months) or outside valid ranges.  
**What we measure:** Count how many records violate each rule.  
**Decision:** For invalid values we set them to missing (`NaN`) so downstream steps can handle them consistently.

In [33]:
invalid_credit_history = (df_clean["financials.credit_history_months"] < 0).sum()
invalid_dti = ((df_clean["financials.debt_to_income"] < 0) | (df_clean["financials.debt_to_income"] > 1)).sum()

print("Invalid credit_history_months (<0):", invalid_credit_history)
print("Invalid debt_to_income (outside [0,1]):", invalid_dti)

Invalid credit_history_months (<0): 2
Invalid debt_to_income (outside [0,1]): 1


In [34]:
df_clean.loc[df_clean["financials.credit_history_months"] < 0, "financials.credit_history_months"] = pd.NA
df_clean.loc[
    (df_clean["financials.debt_to_income"] < 0) | (df_clean["financials.debt_to_income"] > 1),
    "financials.debt_to_income"
] = pd.NA

# quick re-check
print("After fix - invalid credit_history_months:", (df_clean["financials.credit_history_months"] < 0).sum())
print("After fix - invalid debt_to_income:", ((df_clean["financials.debt_to_income"] < 0) | (df_clean["financials.debt_to_income"] > 1)).sum())

After fix - invalid credit_history_months: 0
After fix - invalid debt_to_income: 0


## 7. Duplicate records

**Goal:** Detect and remove duplicated rows that could bias downstream analysis.  
**What we measure:** Number and percentage of exact duplicate rows.  
**Decision:** If duplicates exist, we remove them and report the final dataset size.

In [35]:
# Exclude columns that contain unhashable objects (e.g., lists/dicts like spending_behavior)
dup_subset = [c for c in df_clean.columns if c != "spending_behavior"]

dup_count = df_clean.duplicated(subset=dup_subset).sum()
dup_pct = (dup_count / len(df_clean)) * 100

print("Duplicate rows (excluding spending_behavior):", dup_count)
print("Duplicate rows (%) :", round(dup_pct, 2))

Duplicate rows (excluding spending_behavior): 1
Duplicate rows (%) : 0.2


In [36]:
df_clean = df_clean.drop_duplicates(subset=dup_subset).reset_index(drop=True)
print("Shape after dropping duplicates:", df_clean.shape)

Shape after dropping duplicates: (501, 20)


**Note:** The `spending_behavior` column contains nested structures (lists/dicts), which are not hashable.  
To reliably detect duplicates, we check duplicates on the remaining columns.

## Summary of data quality actions

This notebook applied a structured data quality checklist and produced a cleaned dataset for downstream analysis.

**What was checked and fixed:**
- **Missing values:** computed missing count/% per column; dropped columns with ≥95% missing (`notes`, `financials.annual_salary`)
- **Data types:** enforced numeric/datetime conversions with safe coercion (`errors="coerce"`)
- **Categorical consistency:** standardized `applicant_info.gender` to {"Female", "Male"} (unknown → missing)
- **Date formats:** parsed `applicant_info.date_of_birth` and computed `age` using a fixed reference date (unparseable DOB → missing age)
- **Invalid values:** set logically invalid values to missing (`credit_history_months < 0`, `debt_to_income` outside [0,1])
- **Duplicates:** removed duplicate rows (duplicate check performed excluding `spending_behavior` because it contains nested structures)

**Final output:**
- Saved cleaned dataset to `data/credit_applications_clean.csv`

In [37]:
summary = {
    "rows_final": len(df_clean),
    "cols_final": df_clean.shape[1],
    "missing_age": int(df_clean["age"].isna().sum()) if "age" in df_clean.columns else None,
    "missing_gender": int(df_clean["applicant_info.gender"].isna().sum()) if "applicant_info.gender" in df_clean.columns else None,
}
summary

{'rows_final': 501, 'cols_final': 20, 'missing_age': 162, 'missing_gender': 3}

In [39]:
df_clean.to_csv(
    "../data/credit_applications_clean.csv",
    index=False
)